In [2]:
import bisect

def range_query_1d(sorted_arr, lo, hi):
    l = bisect.bisect_left(sorted_arr, lo)
    r = bisect.bisect_right(sorted_arr, hi)
    return sorted_arr[l:r]


def count_range_1d(sorted_arr, lo, hi):
    l = bisect.bisect_left(sorted_arr, lo)
    r = bisect.bisect_right(sorted_arr, hi)
    return r - l


class RangeIndex1D:
    def __init__(self, items):
        # Sort items by their keys (first element of each tuple)
        self.data = sorted(items, key=lambda x: x[0])
        # Extract just the keys for bisect operations
        self.keys = [item[0] for item in self.data]

    def query(self, lo, hi):
        l = bisect.bisect_left(self.keys, lo)
        r = bisect.bisect_right(self.keys, hi)
        return self.data[l:r]

    def count(self, lo, hi):
        return bisect.bisect_right(self.keys, hi) \
               - bisect.bisect_left(self.keys, lo)


data = [(3,'c'), (1,'a'), (7,'g'),
        (4,'d'),(9,'i'),(2,'b')]

idx = RangeIndex1D(data)
print(idx.query(2,5))
print(idx.count(2,5))


[(2, 'b'), (3, 'c'), (4, 'd')]
3


In [5]:
class SegmentTree:
    def __init__(self, arr, func=sum, identity= 0):
        self.n = len(arr)
        self.fun = func
        self.id = identity
        self.tree = [identity] * (2 * self.n)

        for i, v in enumerate(arr):
            self.tree[self.n + i] = v

        for i in range(self.n - 1 , 0 , -1):
            self.tree[i] = func(
                self.tree[2*i], self.tree[2*i+1]
            )

    def update(self, i, val):
        i += self.n
        self.tree[i] = val
        while i > 1:
            i >>= 1
            self.tree[i] = self.fun(
                self.tree[2*i], self.tree[2*i + 1]
            )

    def query(self, l, r):
        res = self.id
        l += self.n; r += self.n
        while l < r:
            if l & 1:
                res = self.fun(res, self.tree[l]); l += 1
            if r & 1:
                r -= 1; res = self.fun(res, self.tree[r])
            l >>= 1; r >>= 1
        return res



arr = [1,3,5,7,9,11]
st = SegmentTree(arr, func=lambda a,b: a+b, identity= 0)
print(st.query(1,4))
st.update(2,10)
print(st.query(1,4))

st_min = SegmentTree(arr, func=min, identity=float('inf'))
print(st_min.query(2,5))


15
20
5


In [7]:
class BIT:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)

    def update(self, i, delta=1):
        while i <= self.n:
            self.tree[i] += delta
            i += i & (-i)

    def prefix_sum(self, i):
        s = 0
        while i > 0:
            s += self.tree[i]
            i -= i & (-i)
        return s


    def range_sum(self, l, r):
        return self.prefix_sum(r) - self.prefix_sum(l - 1)

    def count_range(self, l, r):
        return self.range_sum(l, r)


bit = BIT(100)
for v in [5,12,33,7,50,21,23,56,78,94]:
    bit.update(v)
print(bit.count_range(5,30))
bit.update(10)
print(bit.count_range(5,30))



class BIT2D:
    def __init__(self, n, m):
        self.tree = [[0]*(m+1) for _ in range(n + 1)]
        self.n, self.m = n, m

    def update(self, x, y, v=1):
        i = x
        while i <= self.n:
            j = y
            while j <= self.m:
                self.tree[i][j] += v
                j += j & -j
            i += i & -i

    def prefix(self, x, y):
        s, i = 0, x
        while i > 0:
            j = y
            while j > 0:
                s += self.tree[i][j]; j -= j & -j
            i -= i & -i
        return s

    def query(self, x1, y1, x2, y2):
        return (self.prefix(x2,y2) - self.prefix(x1 - 1, y2)
        - self.prefix(x2, y1 - 1) + self.prefix(x1 - 1, y1 - 1))


5
6


In [8]:
from collections import namedtuple
KDNode = namedtuple('KDNode',['pt','left','right','axis'])

def build(pts, depth = 0):
    if not pts: return None
    k = len(pts[0])
    axis = depth % k
    pts.sort(key = lambda p: p[axis])
    mid = len(pts) // 2
    return KDNode(pts[mid],
                  build(pts[:mid], depth + 1),
                  build(pts[mid+1:], depth + 1),
                  axis)

def range_searhc_kd(node, lo, hi, result=None):
    if result is None: result = []
    if node is None: return result

    if all(lo[i] <= node.pt[i] <= hi[i]
           for i in range(len(lo))):
       result.append(node.pt)


    axis = node.axis
    if lo[axis] <= node.pt[axis]:
        range_search_kd(node.left, lo, hi, result)
    if hi[axis] >= node.pt[axis]:
        range_search_kd(node.right, lo, hi, result)
    return result


from scipy.spatial import KDTree
import numpy as np

pts = np.random.rand(5000, 2) * 100
tree =KDTree(pts)

idx_r = tree.query_ball_point([50,50], r=10)

lo, hi = np.array([40,40]), np.array([60,60])
mask = np.all((pts >= lo) & (pts <= hi), axis = 1)
idx_rect = np.where(mask)[0]


tree_inf = KDTree(pts)
center = (lo + hi) / 2
half_w = (hi - lo) / 2

pts_n = (pts - center) / half_w
tree_ch = KDTree(pts_n)
idx_ch = tree_ch.query_ball_point(
    [0,0], r = 1.0, p = np.inf
)

print(f"puntos en rectangulo: {len(idx_ch)}")



puntos en rectangulo: 205


In [11]:
import numpy as np
from scipy.spatial import KDTree
from sklearn.neighbors import BallTree

pts = np.random.rand(10000, 2) * 100
tree = KDTree(pts)

query = np.array([50.0,50.0])
idx = tree.query_ball_point(query, r = 5.0)
print(f"dentro de r=5: {len(idx)} puntos")

queries = np.random.rand(50,2) * 100
results = tree.query_ball_point(queries, r=3.0)

coords = np.radians(np.array([
    [19.43, -99.13],
    [20.97, -89.62],
    [25.67, -100.31],
    [29.07, -110.96],
    [32.53, -117.04],

]))

bt = BallTree(coords, metric = 'haversine')

q_rad = np.radians([[19.43,-99.13]])
r_rad = 800 / 6371
idx_geo = bt.query_radius(q_rad, r = r_rad)
print("ciudades <= 800 km: ", idx_geo[0])

def radius_search_brute(pts, query, r, metric= 'euclidean'):
    if metric == 'euclidean':
        dists = np.linalg.norm(pts - query, axis = 1)
    elif metric == 'manhattan':
        dists = np.sum(np.abs(pts - query), axis = 1)
    mask = dists <= r
    return np.where(mask)[0], dists[mask]



from sklearn.cluster import DBSCAN
db = DBSCAN(eps=5.0, min_samples=3,
            algorithm = 'ball_tree',
            metric = 'euclidean').fit(pts)

print(db.labels_)

dentro de r=5: 78 puntos
ciudades <= 800 km:  [0 2]
[0 0 0 ... 0 0 0]


In [13]:
from collections import namedtuple
# KDNode = namedtuple('KDNode',['pt','left','right','axis'])

def point_in_polygon_raycast(point, polygon):
    x, y = point
    n = len(polygon)
    inside = False

    # Iterate over each edge of the polygon
    p1 = polygon[0]
    for i in range(n + 1):
        p2 = polygon[i % n] # This ensures p2 wraps around to polygon[0] for the last edge

        # Check if the ray from `point` (extending rightward) intersects the segment `p1-p2`
        # An intersection occurs if the segment crosses the ray's y-level (point.y)
        if ((p1[1] <= y and p2[1] > y) or (p2[1] <= y and p1[1] > y)):
            # Calculate the x-coordinate of the intersection point with the horizontal ray
            # Avoid division by zero if the segment is perfectly horizontal (p1[1] == p2[1])
            if p1[1] == p2[1]:
                # If the point is on a horizontal segment, it's considered inside if x is between p1[0] and p2[0]
                if y == p1[1] and ((p1[0] <= x <= p2[0]) or (p2[0] <= x <= p1[0])):
                    return True # Point is on the boundary, often considered inside
                continue # No intersection if horizontal segment and point not on it

            x_inters = (y - p1[1]) * (p2[0] - p1[0]) / (p2[1] - p1[1]) + p1[0]

            # If the intersection point is to the right of `point.x`, it's an intersection
            if x < x_inters:
                inside = not inside
        p1 = p2 # Move to the next point

    return inside


def point_in_polygon_winding(point, polygon):
    def is_left(p0, p1, p2):
        return ((p1[0] - p0[0]) * (p2[1] - p0[1]) -
                (p2[0] - p0[0]) * (p1[1] - p0[1]))

    wn = 0    # winding number
    n = len(polygon)
    px, py = point

    for i in range(n):
        p1 = polygon[i]
        p2 = polygon[(i + 1) % n]
        p1x, p1y = p1
        p2x, p2y = p2

        if p1y <= py:  # p1 is below or on the ray
            if p2y > py:  # p2 is above the ray
                if is_left(p1, p2, point) > 0:  # segment crosses ray upward
                    wn += 1
        else:  # p1 is above the ray
            if p2y <= py:  # p2 is below or on the ray
                if is_left(p1, p2, point) < 0:  # segment crosses ray downward
                    wn -= 1
    return wn != 0



poly = [(0,0), (10,0), (10,10), (5,15), (0,10)]

print(point_in_polygon_raycast((5,5), poly))
print(point_in_polygon_raycast((15,15), poly))



from shapely.geometry import Point, Polygon
poly_s = Polygon(poly)
print(poly_s.contains(Point(5,5)))


True
False
True
